In [1]:
%pip install -q h5py torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00


In [2]:
import os
import pickle
import random
import sys
import tarfile
import urllib.request
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.cuda.amp import GradScaler, autocast
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

seed = 42
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
if USE_AMP:
    torch.backends.cudnn.benchmark = True

N_TRAIN = 9000
N_TEST = None
EPOCHS = 5
BATCH_SIZE = 16 if USE_AMP else 4
_cpu = os.cpu_count() or 2
NUM_WORKERS = 0 if sys.platform == "darwin" else min(8, max(0, _cpu - 1))

SVHN_ROOT = Path("data/svhn")
SVHN_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_DIR = SVHN_ROOT / "train"
TEST_DIR = SVHN_ROOT / "test"
STREET_PHOTOS = Path("images/street_numbers")
STREET_PHOTOS.mkdir(parents=True, exist_ok=True)
CHECKPOINT = SVHN_ROOT / "w.pt"

TRAIN_URL = "http://ufldl.stanford.edu/housenumbers/train.tar.gz"
TEST_URL = "http://ufldl.stanford.edu/housenumbers/test.tar.gz"

NUM_CLASSES = 11
LR = 0.035
IOU_THRESH = 0.5
SCORE_THRESH_EVAL = 0.3
FREEZE_BACKBONE = True

In [3]:
def download_and_extract(url: str, root: Path, member_dir: Path):
    name = Path(url).name
    archive = root / name
    if not member_dir.is_dir() or not any(member_dir.iterdir()):
        if not archive.is_file():
            urllib.request.urlretrieve(url, archive)
        with tarfile.open(archive, "r:gz") as tf:
            if hasattr(tarfile, "data_filter"):
                tf.extractall(path=root, filter="data")
            else:
                tf.extractall(path=root)
    else:
        pass


download_and_extract(TRAIN_URL, SVHN_ROOT, TRAIN_DIR)
download_and_extract(TEST_URL, SVHN_ROOT, TEST_DIR)

In [4]:
def _read_bbox_field(f: h5py.File, bbox, key: str):
    dset = bbox[key]
    raw = dset[()]
    vals = []
    for j in range(dset.shape[0]):
        r = raw[j, 0]
        try:
            vals.append(int(np.array(f[r][()]).flat[0]))
        except (TypeError, ValueError, KeyError):
            vals.append(int(r))
    return vals


def parse_digit_struct(mat_path: Path):
    f = h5py.File(mat_path, "r")
    try:
        names = f["digitStruct"]["name"]
        bbs = f["digitStruct"]["bbox"]
        n = names.shape[0]
        out = []
        for i in range(n):
            name_ref = names[i, 0]
            fname = "".join(chr(int(c[0])) for c in f[name_ref][:])
            bb = bbs[i, 0]
            g = f[bb]
            out.append(
                {
                    "file": fname,
                    "label": _read_bbox_field(f, g, "label"),
                    "left": _read_bbox_field(f, g, "left"),
                    "top": _read_bbox_field(f, g, "top"),
                    "width": _read_bbox_field(f, g, "width"),
                    "height": _read_bbox_field(f, g, "height"),
                }
            )
        return out
    finally:
        f.close()


def load_annotations(split_dir: Path, cache_name: str):
    mat = split_dir / "digitStruct.mat"
    cache = split_dir / cache_name
    if cache.is_file():
        with open(cache, "rb") as fp:
            ann = pickle.load(fp)
    else:
        ann = parse_digit_struct(mat)
        with open(cache, "wb") as fp:
            pickle.dump(ann, fp)
    return ann


def subset_ann(ann, max_n, subseed: int):
    if max_n is None or len(ann) <= max_n:
        return ann
    rng = random.Random(subseed)
    idx = list(range(len(ann)))
    rng.shuffle(idx)
    return [ann[i] for i in idx[:max_n]]


_train_full = load_annotations(TRAIN_DIR, "train_ann.pkl")
_test_full = load_annotations(TEST_DIR, "test_ann.pkl")
train_ann = subset_ann(_train_full, N_TRAIN, seed)
test_ann = subset_ann(_test_full, N_TEST, seed + 1)
print(len(train_ann), len(test_ann))

12000 1500


In [15]:
def svhn_label_to_display(lab: int) -> str:
    return "0" if lab == 10 else str(lab)


class SVHNDetectionDataset(Dataset):
    def __init__(self, root: Path, annotations, train: bool):
        self.root = root
        self.annotations = annotations
        self.train = train

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        rec = self.annotations[idx]
        path = self.root / rec["file"]
        img = Image.open(path).convert("RGB")
        w, h = img.size
        boxes = []
        labels = []
        for j in range(len(rec["left"])):
            left = float(rec["left"][j])
            top = float(rec["top"][j])
            bw = float(rec["width"][j])
            bh = float(rec["height"][j])
            x1 = max(0.0, left)
            y1 = max(0.0, top)
            x2 = min(float(w), left + bw)
            y2 = min(float(h), top + bh)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            lab = int(rec["label"][j])
            labels.append(lab)
        if not boxes:
            boxes = [[0.0, 0.0, float(w), float(h)]]
            labels = [1]
        img_t = TF.to_tensor(img)
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
        }
        return img_t, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_ds = SVHNDetectionDataset(TRAIN_DIR, train_ann, train=True)
test_ds = SVHNDetectionDataset(TEST_DIR, test_ann, train=False)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **({} if NUM_WORKERS == 0 else {"num_workers": NUM_WORKERS, "persistent_workers": True, "prefetch_factor": 2}),
    collate_fn=collate_detection,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    **({} if NUM_WORKERS == 0 else {"num_workers": NUM_WORKERS, "persistent_workers": True, "prefetch_factor": 2}),
    collate_fn=collate_detection,
    pin_memory=torch.cuda.is_available(),
)

In [16]:
model = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
in_f = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
if FREEZE_BACKBONE:
    for p in model.backbone.parameters():
        p.requires_grad = False
model = model.to(device)


In [17]:
scaler = GradScaler(enabled=USE_AMP)


def train_one_epoch(model, loader, optim, dev):
    model.train()
    losses_acc = []
    for images, targets in loader:
        images = [im.to(dev) for im in images]
        targets = [{k: v.to(dev) for k, v in t.items()} for t in targets]
        optim.zero_grad(set_to_none=True)
        if USE_AMP:
            with autocast():
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            loss.backward()
            optim.step()
        losses_acc.append(float(loss.detach().float().cpu()))
    return float(np.mean(losses_acc)) if losses_acc else 0.0


if CHECKPOINT.is_file():
    model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
lr_sched = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=sorted({max(1, EPOCHS // 2), max(1, EPOCHS - 1)}), gamma=0.1
)
for ep in range(EPOCHS):
    avg = train_one_epoch(model, train_loader, optimizer, device)
    lr_sched.step()
    print(ep, round(avg, 3))
torch.save(model.state_dict(), CHECKPOINT)


/tmp/ipykernel_9538/800977175.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
/tmp/ipykernel_9538/800977175.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


0 0.804
1 0.726
2 0.711


In [18]:
import torchvision.ops as ops
from torchmetrics.detection.mean_ap import MeanAveragePrecision


def box_iou_xyxy(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return ops.box_iou(a, b)


@torch.no_grad()
def collect_predictions(model, loader, dev, score_thresh: float):
    model.eval()
    preds_tm = []
    targets_tm = []
    total_tp = total_fp = total_fn = 0
    iou_sum = 0.0
    iou_count = 0
    for images, targets in loader:
        imgs = [im.to(dev) for im in images]
        if USE_AMP:
            with autocast():
                outs = model(imgs)
        else:
            outs = model(imgs)
        for o, t in zip(outs, targets):
            keep = o["scores"] >= score_thresh
            pb = o["boxes"][keep].cpu()
            ps = o["scores"][keep].cpu()
            pl = o["labels"][keep].cpu()
            preds_tm.append({"boxes": pb, "scores": ps, "labels": pl})
            gt_boxes = t["boxes"].cpu()
            gt_labels = t["labels"].cpu()
            targets_tm.append({"boxes": gt_boxes, "labels": gt_labels})
            if pb.numel() == 0:
                total_fn += gt_boxes.shape[0]
                continue
            if gt_boxes.numel() == 0:
                total_fp += pb.shape[0]
                continue
            iou = box_iou_xyxy(pb, gt_boxes)
            matched_gt = torch.zeros(gt_boxes.shape[0], dtype=torch.bool)
            matched_pred = torch.zeros(pb.shape[0], dtype=torch.bool)
            order = torch.argsort(ps, descending=True)
            for pi in order.tolist():
                best_j = -1
                best_iou = 0.0
                for gj in range(gt_boxes.shape[0]):
                    if matched_gt[gj]:
                        continue
                    if int(pl[pi]) != int(gt_labels[gj]):
                        continue
                    v = float(iou[pi, gj])
                    if v > best_iou:
                        best_iou = v
                        best_j = gj
                if best_j >= 0 and best_iou >= IOU_THRESH:
                    matched_pred[pi] = True
                    matched_gt[best_j] = True
                    total_tp += 1
                    iou_sum += best_iou
                    iou_count += 1
            total_fp += int((~matched_pred).sum())
            total_fn += int((~matched_gt).sum())
    return preds_tm, targets_tm, total_tp, total_fp, total_fn, iou_sum, iou_count


preds, tgts, tp, fp, fn, iou_sum, iou_n = collect_predictions(
    model, test_loader, device, SCORE_THRESH_EVAL
)
prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
mean_iou = iou_sum / iou_n if iou_n > 0 else 0.0
map_metric = MeanAveragePrecision(box_format="xyxy", class_metrics=False)
map_metric.update(preds, tgts)
m = map_metric.compute()
v50 = m.get("map_50")
map50 = float(v50) if v50 is not None and float(v50) >= 0 else float(m["map"])
print(round(mean_iou, 4), round(prec, 4), round(rec, 4), round(map50, 4))

/tmp/ipykernel_9538/4175379695.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


0.7258 0.5661 0.6875 0.6222
